In [ ]:
!pip install langchain

In [1]:
from langchain import PromptTemplate

template = '''
Question: {question}
Answer: 
'''

prompt = PromptTemplate(
    template = template,
    input_variables = ['question']
)
prompt

PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='\nQuestion: {question}\nAnswer: \n')

In [ ]:
import os
os.environ['HUGGINGFACEHUB_API_TOKEN'] = ''

In [ ]:
!pip install langchain-huggingface

In [3]:
# Initialize the Hugging Face Inference Client
from huggingface_hub import InferenceClient

client = InferenceClient(
    model = "HuggingFaceH4/zephyr-7b-alpha", 
)

In [8]:
from langchain_core.output_parsers import StrOutputParser

# create the chain
llm_chain = prompt | hub_llm | StrOutputParser()

In [ ]:
qn = "Who is Elon Musk?"
print(llm_chain.invoke(qn))

In [ ]:
qn = "Is he married?"
print(llm_chain.invoke(qn))

In [ ]:
template = '''
Current conversation: {history}
Human: {question}
AI:
'''

In [ ]:
prompt = PromptTemplate(
    template = template,
    input_variables = ['question','history']

In [ ]:
# create the chain
llm_chain = prompt | hub_llm | StrOutputParser()

In [ ]:
qn = "Who is Elon Musk?"
response = llm_chain.invoke({'question':qn,'history':''})
print(response)

In [ ]:
qn = "Is he married?"
response = llm_chain.invoke({'question':qn,'history':response})
print(response)

In [ ]:
history = ''
while True:    
    qn = input('Question: ')
    if qn == 'quit':
        break        
    response = llm_chain.invoke({'question':qn, 'history':history})
    history = response
    print(history)

In [ ]:
import os
from langchain import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import HuggingFaceEndpoint
from langchain_core.runnables.history import RunnableWithMessageHistory

os.environ['HUGGINGFACEHUB_API_TOKEN'] = 'your_hugging_face_token'

template = '''
Question: {question}
Answer: 
'''

prompt = PromptTemplate(
    template = template,
    input_variables = ['question']
)

hub_llm = HuggingFaceEndpoint(
    endpoint_url="https://api-inference.huggingface.co/models/HuggingFaceH4/zephyr-7b-alpha", 
    temperature = 1
)

class SessionHistory:  
    def __init__(self):
        self.messages = []

    def add_messages(self, messages):
        self.messages.extend(messages)  

    def get_messages(self):
        return self.messages

session_history = SessionHistory()  

def get_session_history():  
    return session_history

llm_chain = RunnableWithMessageHistory(  
    prompt | hub_llm | StrOutputParser(),
    get_session_history = get_session_history
)

while True:  
    # get the user input
    user_question = input("Ask a question (type 'exit' to stop): ")

    # exit condition
    if user_question.lower() == "quit":
        print("Ending conversation.")
        break

    # prepare the input data
    input_data = {"question": user_question}

    response = llm_chain.invoke(input_data)  

    session_history.add_messages([  
        {"role": "user", "content": user_question},
        {"role": "assistant", "content": response}
    ])

    # display the response
    print(f"AI: {response}")

In [ ]:
import os
from langchain import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import HuggingFaceEndpoint

os.environ['HUGGINGFACEHUB_API_TOKEN'] = 'Your_HuggingFace_Token'

template = '''
Complete this: {question}
'''

prompt = PromptTemplate(
    template = template,
    input_variables = ['question']
)
prompt

hub_llm = HuggingFaceEndpoint(
    endpoint_url="https://api-inference.huggingface.co/models/HuggingFaceH4/zephyr-7b-alpha", 
    temperature = 1
)

llm_chain = prompt | hub_llm | StrOutputParser()

while True:    
    qn = input('Question: ')
    if qn == 'quit':
        break        
    response = llm_chain.invoke(qn)
    print(response)

In [ ]:
from langchain.chat_models import ChatOpenAI

openai_model = ChatOpenAI(model_name = 'gpt-4o-mini')

In [ ]:
from langchain import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

template = '''
Question: {question}
Answer: '''

prompt = PromptTemplate(
    template = template,
    input_variables = ['question']
)

llm_chain = prompt | openai_model | StrOutputParser()

In [ ]:
question = "Who is Steve Jobs"
print(llm_chain.invoke(question))

In [ ]:
import os
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import HuggingFaceEndpoint
from langchain import PromptTemplate

os.environ['HUGGINGFACEHUB_API_TOKEN'] = 'Your_HuggingFace_Token'

template = '''
Question: {question}
Answer: '''

prompt = PromptTemplate(
    template = template,
    input_variables = ['question']
)

hub_llm = HuggingFaceEndpoint(endpoint_url=
    "https://api-inference.huggingface.co/models/tiiuae/falcon-7b-instruct", 
    temperature = 1
)

llm_chain = prompt | hub_llm | StrOutputParser()

In [ ]:
question = "Translate this to Spanish: Which is the way to the train station?"
print(llm_chain.invoke(question))

In [ ]:
question = "What is the capital of France?"
print(llm_chain.invoke(question))

In [ ]:
!pip install llama_index
!pip install llama-index-embeddings-huggingface
!pip install llama-index-llms-huggingface

In [ ]:
from llama_index.core import SimpleDirectoryReader

loader = SimpleDirectoryReader(
    input_dir="./Training Documents",
    recursive=True,
    required_exts=[".pdf"],
)

# loads the documents
documents = loader.load_data()

In [ ]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embedding_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

In [ ]:
from llama_index.core import VectorStoreIndex

index = VectorStoreIndex.from_documents(
    documents,
    embed_model = embedding_model,
)

# save the index in the current directory
index.storage_context.persist(persist_dir=".")

In [ ]:
from llama_index.core import StorageContext, load_index_from_storage
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

embedding_model = HuggingFaceEmbedding(model_name="BAAI/bge-small-en-v1.5")

storage_context = StorageContext.from_defaults(persist_dir=".")
index = load_index_from_storage(storage_context,
                                embed_model = embedding_model)

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from llama_index.llms.huggingface import HuggingFaceLLM
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
tokenizer = AutoTokenizer.from_pretrained(
    "meta-llama/Llama-3.2-3B-Instruct")
    model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.2-3B-Instruct").to(device)

    huggingface_llm = HuggingFaceLLM(
        model=model,
        tokenizer=tokenizer,
    )

    # set the LLM to use
    query_engine = index.as_query_engine(llm=huggingface_llm)

In [ ]:
while True:
    question = input("Question: ")
    if question.lower() == "quit": break
    print(query_engine.query(question).response)

In [ ]:
!pip install langchain_community
!pip install langchain_openai

In [ ]:
from langchain_openai import ChatOpenAI
import os 

os.environ["OPENAI_API_KEY"] = ""
openai_llm = ChatOpenAI(temperature = 0.7, 
                        model_name = "gpt-4o-mini")

query_engine = index.as_query_engine(llm = openai_llm)

In [ ]:
while True:
    question = input("Question: ")
    if question.lower() == "quit": break
    print(query_engine.query(question).response)

In [ ]:
!pip install gradio

In [ ]:
def my_chat_bot(input_text):
    response = query_engine.query(input_text)
    return response.response

In [ ]:
import gradio as gr

# bind it to gradio
gr.Interface(fn = my_chat_bot, 
             title = "Enquiry",
             inputs = "text", 
             outputs = "text").launch()

In [ ]:
# create a query engine to ask question
query_engine = index.as_chat_engine(llm=openai_llm)

In [ ]:
def my_chat_bot(input_text):
    response = query_engine.chat(input_text)  
    return response.response

import gradio as gr

gr.Interface(fn = my_chat_bot,  
             title = "Enquiry",
             inputs = "text", 
             outputs = "text").launch()


In [ ]:
import gradio as gr

with gr.Blocks() as mychatbot:  
    chatbot = gr.Chatbot()  #A
    question = gr.Textbox()  #B
    
    def chat(message, chat_history):
        content = "Responses from chatbot..." 
        chat_history.append((message, content))
        return "", chat_history
    
    question.submit(fn = chat, 
                    inputs = [question, chatbot],
                    outputs = [question, chatbot])

mychatbot.launch()

In [ ]:
import gradio as gr

with gr.Blocks() as mychatbot:
    chatbot = gr.Chatbot()  
    question = gr.Textbox() 
    
    def chat(message, chat_history):
        content = my_chat_bot(message)
        chat_history.append((message, content))
        return "", chat_history
  
    question.submit(fn = chat,
                    inputs = [question, chatbot],
                    outputs = [question, chatbot])

mychatbot.launch()